# 📚 Multi-Source Academic PPTX Scraper — Colab V3

Scrapes PPT/PPTX from **3 major open-access repositories** into one Google Drive folder:

| Source | API | Estimated PPT files |
|--------|-----|--------------------|
| **Figshare** | v2 POST search | ~800-1000 |
| **Zenodo** | REST search (16k+ records) | ~2000-5000 |
| **HAL** (France) | Solr search | ~500-1500 |

**Filters:** ❌ China/HK/Taiwan | ❌ USA | ❌ Under 2MB | ❌ Duplicates | ✅ Magic bytes

In [ ]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')
import os, warnings
warnings.filterwarnings('ignore')
DRIVE_DIR = '/content/drive/MyDrive/PPTX_Academic_Download'
SEEN_FILE = os.path.join(DRIVE_DIR, 'seen_tags.txt')
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'✅ Output: {DRIVE_DIR}')

In [ ]:
# Cell 2: Shared utilities
import hashlib, json, logging, os, re, time, zipfile
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from urllib.parse import unquote
logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('MultiScraper')
MIN_SIZE = 2 * 1024 * 1024

CHINA_KW = ['chinese','china','hong kong','taiwan','beijing','shanghai',
            'mandarin','wuhan','guangzhou','shenzhen','nanjing','zhonghua']
USA_KW = ['american government','american history','us constitution',
          'us election','us president','fourth of july','veterans day',
          'memorial day','pledge of allegiance','state of the union']

def load_seen():
    tags = set()
    if os.path.exists(SEEN_FILE):
        with open(SEEN_FILE) as f:
            tags = {l.strip() for l in f if l.strip()}
    for fn in os.listdir(DRIVE_DIR):
        if fn.endswith(('.ppt','.pptx')) and '_' in fn:
            t = fn.split('_')[0]
            if len(t) == 10: tags.add(t)
    return tags

def save_tag(tag):
    with open(SEEN_FILE, 'a') as f: f.write(tag + '\n')

def is_blocked(text):
    t = text.lower()
    return any(kw in t for kw in CHINA_KW) or any(kw in t for kw in USA_KW)

def has_chinese_chars(fp):
    try:
        with zipfile.ZipFile(fp, 'r') as z:
            for n in z.namelist():
                if 'slide' in n and n.endswith('.xml'):
                    d = z.read(n).decode('utf-8', errors='ignore')
                    if len(re.findall(r'[\u4e00-\u9fff]', d)) > 50: return True
    except: pass
    return False

def valid_ppt(fp):
    try:
        with open(fp, 'rb') as f: h = f.read(8)
        return h[:2] == b'PK' or h[:8] == b'\xd0\xcf\x11\xe0\xa1\xb1\x1a\xe1'
    except: return False

class BaseDownloader:
    def __init__(self):
        self.s = requests.Session()
        retry = Retry(total=3, backoff_factor=1, status_forcelist=[429,500,502,503,504])
        self.s.mount('https://', HTTPAdapter(max_retries=retry))
        self.s.mount('http://', HTTPAdapter(max_retries=retry))
        self.s.headers.update({'User-Agent':'Mozilla/5.0 Academic-Scraper/3.0'})
        self.seen = load_seen()
        self.count = 0
        self.st = {'checked':0,'found':0,'china':0,'usa':0,'small':0,
                   'invalid':0,'dup':0,'downloaded':0,'errors':0}
        print(f'Loaded {len(self.seen)} previously seen tags')

    def download_file(self, url, title, source_prefix, source_id, filename):
        tag = hashlib.sha1(url.encode()).hexdigest()[:10]
        if tag in self.seen: self.st['dup'] += 1; return False
        if is_blocked(title) or is_blocked(filename):
            k = 'china' if any(kw in title.lower() for kw in CHINA_KW) else 'usa'
            self.st[k] += 1; return False

        ext = '.ppt' if filename.lower().endswith('.ppt') else '.pptx'
        safe = re.sub(r'[^\w\-]', '_', title[:45])
        fname = f'{tag}_{source_prefix}_{source_id}_{safe}{ext}'
        dest = os.path.join(DRIVE_DIR, fname)
        if os.path.exists(dest): self.seen.add(tag); return False

        try:
            try:
                h = self.s.head(url, timeout=10, allow_redirects=True)
                cl = int(h.headers.get('Content-Length', 0))
                if 0 < cl < MIN_SIZE: self.st['small'] += 1; return False
            except: pass

            r = self.s.get(url, timeout=90, stream=True)
            if not r.ok: self.st['errors'] += 1; return False
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(65536): f.write(chunk)

            sz = os.path.getsize(dest)
            if sz < MIN_SIZE: os.remove(dest); self.st['small'] += 1; return False
            if not valid_ppt(dest): os.remove(dest); self.st['invalid'] += 1; return False
            if has_chinese_chars(dest): os.remove(dest); self.st['china'] += 1; return False

            self.seen.add(tag); save_tag(tag)
            self.count += 1; self.st['downloaded'] += 1
            logger.info(f'✅ [{self.count}] {fname} ({sz//1024}KB)')
            return True
        except:
            if os.path.exists(dest): os.remove(dest)
            self.st['errors'] += 1; return False

print('✅ Base engine ready')

In [ ]:
# Cell 3: FIGSHARE source
FIGSHARE_SEARCH = 'https://api.figshare.com/v2/articles/search'
FIGSHARE_FILES  = 'https://api.figshare.com/v2/articles/{}/files'
FIG_TYPES = [None, 7, 5, 3, 2, 8, 6, 12]

FIG_QUERIES = [
    '.pptx', '.ppt', 'powerpoint', 'presentation slides',
    'lecture slides', 'slide deck', 'conference presentation',
    'workshop slides', 'tutorial presentation', 'seminar slides',
    'machine learning', 'artificial intelligence', 'deep learning',
    'climate change', 'renewable energy', 'public health',
    'genetics', 'genomics', 'biology', 'ecology', 'chemistry', 'physics',
    'computer science', 'data science', 'engineering',
    'economics', 'finance', 'psychology', 'neuroscience',
    'education', 'sociology', 'agriculture', 'pharmacology',
    'environmental science', 'geology', 'robotics', 'cybersecurity',
    'nanotechnology', 'astronomy', 'biodiversity', 'conservation',
    'water resources', 'covid', 'pandemic', 'cancer', 'quantum',
]

def scrape_figshare(dl, target=2000):
    print('\n' + '='*60)
    print('🔬 PHASE 1: FIGSHARE')
    print('='*60)
    seen_articles = set()
    start_count = dl.count

    for qi, q in enumerate(FIG_QUERIES):
        if dl.count - start_count >= target: break
        print(f'  [{qi+1}/{len(FIG_QUERIES)}] "{q}" | Total: {dl.count}')

        for itype in FIG_TYPES:
            if dl.count - start_count >= target: break
            page = 1
            while page <= 20:
                if dl.count - start_count >= target: break
                payload = {'search_for':q,'page_size':50,'page':page,
                           'order':'published_date','order_direction':'desc'}
                if itype is not None: payload['item_type'] = itype
                try:
                    r = dl.s.post(FIGSHARE_SEARCH, json=payload, timeout=30)
                    articles = r.json() if r.ok and r.status_code != 422 else []
                except: articles = []
                if not articles: break
                dl.st['checked'] += len(articles)

                for art in articles:
                    aid = art.get('id')
                    if aid in seen_articles: continue
                    seen_articles.add(aid)
                    title = art.get('title','untitled')
                    try:
                        fr = dl.s.get(FIGSHARE_FILES.format(aid), timeout=30)
                        files = [f for f in fr.json() if f.get('name','').lower().endswith(('.ppt','.pptx')) and f.get('download_url')] if fr.ok else []
                    except: files = []
                    dl.st['found'] += len(files)
                    for f in files:
                        dl.download_file(f['download_url'], title, 'fig', str(aid), f.get('name','f.pptx'))
                    time.sleep(0.2)
                page += 1
                time.sleep(0.3)
        time.sleep(0.5)
    print(f'  Figshare done: {dl.count - start_count} new files')

print('✅ Figshare module ready')

In [ ]:
# Cell 4: ZENODO source (16,490+ PPT records!)
ZENODO_API = 'https://zenodo.org/api/records'

ZEN_QUERIES = [
    # Direct filename searches — highest yield
    'filename:*.pptx', 'filename:*.ppt',
    # Filename + topic combos
    'filename:*.pptx climate', 'filename:*.pptx health',
    'filename:*.pptx machine learning', 'filename:*.pptx biology',
    'filename:*.pptx education', 'filename:*.pptx engineering',
    'filename:*.pptx environment', 'filename:*.pptx social',
    'filename:*.pptx economics', 'filename:*.pptx chemistry',
    'filename:*.pptx physics', 'filename:*.pptx computer',
    'filename:*.pptx conference', 'filename:*.pptx workshop',
    'filename:*.pptx lecture', 'filename:*.pptx seminar',
    'filename:*.pptx research', 'filename:*.pptx university',
    'filename:*.pptx data', 'filename:*.pptx analysis',
    'filename:*.pptx methodology', 'filename:*.pptx survey',
    'filename:*.pptx experiment', 'filename:*.pptx results',
    'filename:*.pptx study', 'filename:*.pptx review',
    'filename:*.pptx report', 'filename:*.pptx design',
    'filename:*.pptx model', 'filename:*.pptx simulation',
    'filename:*.pptx project', 'filename:*.pptx development',
    'filename:*.pptx training', 'filename:*.pptx assessment',
    'filename:*.pptx policy', 'filename:*.pptx management',
    'filename:*.pptx technology', 'filename:*.pptx innovation',
    'filename:*.pptx agriculture', 'filename:*.pptx medicine',
    'filename:*.pptx ecology', 'filename:*.pptx genetics',
    'filename:*.pptx astronomy', 'filename:*.pptx geology',
    'filename:*.pptx robotics', 'filename:*.pptx energy',
    'filename:*.pptx water', 'filename:*.pptx food',
    'filename:*.pptx transport', 'filename:*.pptx security',
    'filename:*.pptx materials', 'filename:*.pptx nano',
    # .ppt variants
    'filename:*.ppt conference', 'filename:*.ppt workshop',
    'filename:*.ppt lecture', 'filename:*.ppt research',
    'filename:*.ppt health', 'filename:*.ppt science',
    'filename:*.ppt engineering', 'filename:*.ppt education',
    # Resource type filter
    'filename:*.pptx resource_type.type:presentation',
    'filename:*.pptx resource_type.type:lesson',
    'filename:*.pptx resource_type.type:other',
]

def extract_zenodo_ppts(record):
    ppts = []
    files_obj = record.get('files')
    if isinstance(files_obj, list):
        for e in files_obj:
            key = e.get('key','')
            if key.lower().endswith(('.ppt','.pptx')):
                lnk = e.get('links',{})
                url = lnk.get('self','')
                if url and not url.endswith('/content'): url += '/content'
                if url: ppts.append({'key':key,'url':url})
    elif isinstance(files_obj, dict):
        entries = files_obj.get('entries',{})
        items = entries.items() if isinstance(entries,dict) else []
        for k, e in items:
            key = e.get('key', k)
            if key.lower().endswith(('.ppt','.pptx')):
                lnk = e.get('links',{})
                url = lnk.get('self','')
                if url and not url.endswith('/content'): url += '/content'
                if url: ppts.append({'key':key,'url':url})
    return ppts

def scrape_zenodo(dl, target=5000):
    print('\n' + '='*60)
    print('🌐 PHASE 2: ZENODO')
    print('='*60)
    seen_records = set()
    start_count = dl.count

    for qi, q in enumerate(ZEN_QUERIES):
        if dl.count - start_count >= target: break
        print(f'  [{qi+1}/{len(ZEN_QUERIES)}] "{q}" | Total: {dl.count}')
        page = 1

        while page <= 40 and dl.count - start_count < target:
            params = {'q':q,'sort':'mostrecent','page':page,'size':50,'access_right':'open'}
            try:
                r = dl.s.get(ZENODO_API, params=params, timeout=30)
                data = r.json() if r.ok else {}
            except: data = {}
            hits = data.get('hits',{}).get('hits',[])
            if not hits: break
            dl.st['checked'] += len(hits)

            for rec in hits:
                rid = rec.get('id')
                if rid in seen_records: continue
                seen_records.add(rid)
                meta = rec.get('metadata',{})
                title = meta.get('title','untitled')
                ppts = extract_zenodo_ppts(rec)
                dl.st['found'] += len(ppts)
                for p in ppts:
                    if dl.count - start_count >= target: break
                    dl.download_file(p['url'], title, 'zen', str(rid), p['key'])
                time.sleep(0.2)
            page += 1
            time.sleep(0.5)
    print(f'  Zenodo done: {dl.count - start_count} new files')

print('✅ Zenodo module ready')

In [ ]:
# Cell 5: HAL (French Open Archives) source
HAL_API = 'https://api.archives-ouvertes.fr/search/'

HAL_QUERIES = [
    'conference presentation', 'slides conférence', 'séminaire exposé',
    'workshop tutorial', 'lecture cours', 'symposium colloque',
    'powerpoint pptx', 'présentation scientifique',
    'research presentation', 'talk science',
    'journée étude', 'school summer winter',
    'invited talk keynote', 'communications poster', 'colloque journée',
    # English academic terms
    'machine learning', 'artificial intelligence', 'climate change',
    'renewable energy', 'public health', 'biology', 'chemistry',
    'physics', 'mathematics', 'engineering', 'computer science',
    'data science', 'economics', 'education',
    # French academic terms
    'apprentissage automatique', 'intelligence artificielle',
    'changement climatique', 'énergie renouvelable',
    'santé publique', 'biologie', 'chimie', 'physique',
    'informatique', 'ingénierie', 'économie',
]

def scrape_hal(dl, target=2000):
    print('\n' + '='*60)
    print('🇫🇷 PHASE 3: HAL (French Archives)')
    print('='*60)
    start_count = dl.count
    seen_hal = set()

    for qi, q in enumerate(HAL_QUERIES):
        if dl.count - start_count >= target: break
        print(f'  [{qi+1}/{len(HAL_QUERIES)}] "{q}" | Total: {dl.count}')
        start = 0

        while start < 5000 and dl.count - start_count < target:
            params = {'q':q, 'fl':'halId_s,title_s,files_s',
                      'wt':'json', 'rows':100, 'start':start,
                      'sort':'producedDate_tdate desc'}
            try:
                r = dl.s.get(HAL_API, params=params, timeout=30)
                data = r.json() if r.ok else {}
            except: data = {}
            docs = data.get('response',{}).get('docs',[])
            if not docs: break
            dl.st['checked'] += len(docs)

            for doc in docs:
                hal_id = doc.get('halId_s','')
                if hal_id in seen_hal: continue
                seen_hal.add(hal_id)
                titles = doc.get('title_s') or ['untitled']
                title = titles[0] if isinstance(titles,list) else str(titles)
                files_s = doc.get('files_s') or []
                if isinstance(files_s,str): files_s = [files_s]

                for furl in files_s:
                    path = furl.split('?')[0].lower()
                    if path.endswith(('.ppt','.pptx')):
                        dl.st['found'] += 1
                        ext_name = os.path.basename(unquote(furl.split('?')[0]))
                        dl.download_file(furl, title, 'hal', hal_id.replace('/','_'), ext_name or 'file.pptx')
                time.sleep(0.2)
            start += 100
            time.sleep(0.3)
    print(f'  HAL done: {dl.count - start_count} new files')

print('✅ HAL module ready')

In [ ]:
# Cell 6: RUN ALL THREE!
dl = BaseDownloader()
print(f'\n🚀 Starting multi-source scrape...')
print(f'   Sources: Figshare → Zenodo → HAL')
print(f'   Output: {DRIVE_DIR}')

scrape_figshare(dl, target=2000)
scrape_zenodo(dl, target=5000)
scrape_hal(dl, target=2000)

print('\n' + '='*60)
print('📊 COMBINED RESULTS')
print('='*60)
for k,v in dl.st.items():
    print(f'   {k:20s}: {v}')
print(f'   {"TOTAL DOWNLOADED":20s}: {dl.count}')
print('='*60)

from datetime import datetime
with open(os.path.join(DRIVE_DIR, f'stats_{datetime.now().strftime("%Y%m%d_%H%M")}.json'), 'w') as f:
    json.dump({**dl.st, 'total':dl.count}, f, indent=2)

In [ ]:
# Cell 7: Summary
import glob
files = glob.glob(os.path.join(DRIVE_DIR,'*.pptx')) + glob.glob(os.path.join(DRIVE_DIR,'*.ppt'))
total_gb = sum(os.path.getsize(f) for f in files) / (1024**3)
fig = len([f for f in files if '_fig_' in f])
zen = len([f for f in files if '_zen_' in f])
hal = len([f for f in files if '_hal_' in f])
print(f'📊 Final Summary:')
print(f'   Figshare: {fig} files')
print(f'   Zenodo:   {zen} files')
print(f'   HAL:      {hal} files')
print(f'   Total:    {len(files)} files | {total_gb:.2f} GB')